# Cutoff Selection

Choosing the seven forecast origins the whole evaluation rests on: **4 event-driven
and 3 quiet**.

**Target series:** MPOB official daily crude palm oil price ("Local Delivered"),
weekly. A physical transaction price, not a futures contract, so it carries no
monthly roll artifact to correct for -- unlike the Yahoo `CPO=F` series this project
used before MPOB was cleared for local use. See [`DATA.md`](DATA.md) for the full
three-source comparison and the data-governance rule this notebook operates under.

The point of the event/quiet split is to separate two questions. On event cutoffs,
does reading news let an agent anticipate a shock a statistical baseline cannot see?
On quiet cutoffs, does the agent *avoid damaging* a forecast when there is nothing to
react to? A method that only wins on shocks and loses on calm weeks is not useful.

Everything here is derived from data, with the search shown rather than only its
result -- including where an easier, worse-quality answer was available and rejected.


---
## 1. Setup


In [1]:
from __future__ import annotations

import itertools
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv


ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(ROOT / "implementations"))
load_dotenv(ROOT / ".env")

from cpo.data import MPOB_WEEKLY_SERIES_ID, build_mpob_service
from cpo.plots import DEFAULT_CUTOFFS, HORIZONS_WEEKS, plot_cutoff_windows, plot_price_history


svc = build_mpob_service(cache_dir=ROOT / "data" / "mpob")
as_of = datetime.now(tz=timezone.utc).replace(tzinfo=None)

weekly = svc.get_series(MPOB_WEEKLY_SERIES_ID, as_of=as_of).set_index("timestamp")["value"]
returns = weekly.pct_change() * 100

news = pd.read_csv(ROOT / "implementations" / "cpo" / "palm_articles_daily.csv")
news["date"] = pd.to_datetime(news["date"], errors="coerce")
news["article_count"] = pd.to_numeric(news["article_count"], errors="coerce")
n_bad = news["date"].isna().sum()
news = news.dropna(subset=["date", "article_count"])
news_weekly = news.set_index("date")["article_count"].resample("W-FRI").sum()

MAX_HORIZON = max(HORIZONS_WEEKS)
print(f"{len(weekly)} weekly prices, {weekly.index.min():%Y-%m-%d} -> {weekly.index.max():%Y-%m-%d}")
print(f"horizons: {HORIZONS_WEEKS} weeks (longest {MAX_HORIZON})")
print(f"dropped {n_bad} malformed rows from the news CSV (embedded commas/newlines in the text column)")

971 weekly prices, 2008-01-04 -> 2026-08-07
horizons: [1, 2, 4, 8, 13] weeks (longest 13)
dropped 18 malformed rows from the news CSV (embedded commas/newlines in the text column)


---
## 2. The four constraints

| # | Constraint | Why |
|---|---|---|
| 1 | Cutoff ≥ 2024-02 | GDELT news starts 2024-01; leave 4 weeks of prior context |
| 2 | Every horizon resolves | A 13-week horizon needs 13 weeks of realised prices after the cutoff |
| 3 | ≥ 100 articles in the prior 8 weeks | An agent needs news to reason over, or the comparison is empty |
| 4 | ≥ 10 weeks between all seven cutoffs | Non-overlapping windows keep the seven scores independent |

Constraint 3 matters more than it looks — GDELT coverage is uneven (see
[`01_cpo_data_exploration.ipynb`](01_cpo_data_exploration.ipynb) §5), and a cutoff in
a sparse stretch would test nothing.


In [2]:
def summarise(cutoff: pd.Timestamp) -> dict | None:
    """Return forward-looking stats for a candidate cutoff, or None if it cannot resolve."""
    targets = [cutoff + pd.Timedelta(weeks=h) for h in HORIZONS_WEEKS]
    if any(t not in weekly.index for t in targets):
        return None
    forward = returns.loc[cutoff + pd.Timedelta(weeks=1) : cutoff + pd.Timedelta(weeks=MAX_HORIZON)]
    return {
        "cutoff": cutoff,
        "price": float(weekly[cutoff]),
        "max_move": float(forward.abs().max()),
        "total_13wk": float(weekly[targets[-1]] / weekly[cutoff] - 1) * 100,
        "news_8wk": float(news_weekly.loc[cutoff - pd.Timedelta(weeks=8) : cutoff].sum()),
        "year": cutoff.year,
    }


pool = pd.DataFrame(
    [s for c in weekly.index if c >= pd.Timestamp("2024-02-01") and (s := summarise(c)) and s["news_8wk"] >= 100]
)
print(f"weeks in the GDELT window        : {(weekly.index >= '2024-02-01').sum()}")
print(f"...that resolve at every horizon and clear the news floor: {len(pool)}")
print(f"latest usable cutoff             : {pool.cutoff.max():%Y-%m-%d}")

print("\nlargest available move per year:")
print(pool.groupby("year").max_move.apply(lambda x: x.nlargest(3).round(2).tolist()))

weeks in the GDELT window        : 132
...that resolve at every horizon and clear the news floor: 79
latest usable cutoff             : 2026-05-08

largest available move per year:
year
2024    [7.72, 7.72, 7.72]
2025    [7.07, 7.07, 7.07]
2026      [2.99, 2.4, 2.4]
Name: max_move, dtype: object


2026 tops out around 3% — there is no large, news-covered move available that year.
That constrains what a fully independent set can include; see Section 3.


---
## 3. Searching for a spaced, well-separated set

Take the 20 largest-move candidates and search every combination of 4 for one where
all four sit ≥10 weeks apart. A greedy pick (always take the next-largest move) can
miss a valid combination that exists further down the ranking — exhaustive search
does not.


In [3]:
def spaced(dates: list[pd.Timestamp], min_gap_days: int = 70) -> bool:
    d = sorted(dates)
    return all((d[i + 1] - d[i]).days >= min_gap_days for i in range(len(d) - 1))


top20 = pool.sort_values("max_move", ascending=False).head(20)
n_spaced = sum(1 for combo in itertools.combinations(top20.itertuples(), 4) if spaced([x.cutoff for x in combo]))
print(f"4-of-20 combinations mutually >=10 weeks apart: {n_spaced}")

distinct = set()
for combo in itertools.combinations(top20.itertuples(), 4):
    dates = [x.cutoff for x in combo]
    if spaced(dates):
        distinct.add(tuple(sorted(d.date() for d in dates)))
print(f"distinct valid combinations (top 20 pool): {len(distinct)}")
for c in sorted(distinct):
    print(" ", [str(d) for d in c])

4-of-20 combinations mutually >=10 weeks apart: 3
distinct valid combinations (top 20 pool): 3
  ['2024-02-02', '2024-04-12', '2024-07-26', '2024-10-04']
  ['2024-02-02', '2024-04-12', '2024-07-26', '2024-10-11']
  ['2024-02-02', '2024-04-12', '2024-07-26', '2024-10-18']


Unlike the Yahoo-based search (which found **zero** valid combinations here), MPOB's
top-20 pool does contain independent options — but they are all effectively the same
solution: four dates in Feb–Oct 2024, wobbling by a week on the last one. Nothing from
2025 or 2026 makes the top-20 cut.

**Widen the pool and require at least 2 distinct years**, trading a small amount of
raw move size for temporal spread — a cutoff in 2025, past more of what an LLM's
training data is likely to have seen, is worth a modest concession on separation.


In [4]:
top30 = pool.sort_values("max_move", ascending=False).head(30)

best = None
for combo in itertools.combinations(top30.itertuples(), 4):
    dates = [x.cutoff for x in combo]
    if not spaced(dates) or len({d.year for d in dates}) < 2:
        continue
    min_event = min(x.max_move for x in combo)
    remaining = pool[~pool.cutoff.isin(dates)].sort_values("max_move")
    quiet_dates: list[pd.Timestamp] = []
    for row in remaining.itertuples():
        if all(abs((row.cutoff - c).days) >= 70 for c in dates + quiet_dates):
            quiet_dates.append(row.cutoff)
            if len(quiet_dates) == 3:
                break
    if len(quiet_dates) < 3:
        continue
    quiet_moves = [pool[pool.cutoff == c].max_move.iloc[0] for c in quiet_dates]
    clean = min_event > max(quiet_moves)
    separation = min_event / max(quiet_moves)
    if best is None or (clean, separation) > (best[0], best[1]):
        best = (clean, separation, dates, quiet_dates, min_event, max(quiet_moves))

clean, separation, event_dates, quiet_dates, min_event, max_quiet = best
print(
    f"best: cleanly ordered={clean}  separation={separation:.2f}x  "
    f"(weakest event {min_event:.2f}% vs strongest quiet {max_quiet:.2f}%)"
)

best: cleanly ordered=True  separation=1.82x  (weakest event 7.07% vs strongest quiet 3.89%)


---
## 4. The selected seven


In [5]:
events = pool[pool.cutoff.isin(event_dates)].copy()
events["kind"] = "event"
quiet = pool[pool.cutoff.isin(quiet_dates)].copy()
quiet["kind"] = "quiet"

selected = pd.concat([events, quiet]).sort_values("cutoff").reset_index(drop=True)
selected["cutoff_str"] = selected.cutoff.dt.strftime("%Y-%m-%d")
selected[["cutoff_str", "kind", "price", "max_move", "total_13wk", "news_8wk"]].round(1)

,cutoff_str,kind,price,max_move,total_13wk,news_8wk
0,2024-02-02,event,3800.0,7.7,2.1,150.0
1,2024-04-19,quiet,4100.0,3.9,-2.3,226.0
2,2024-07-26,event,4029.5,7.1,16.3,165.0
3,2024-10-18,event,4374.0,7.1,4.7,162.0
4,2025-01-17,event,4577.5,7.1,-8.7,257.0
5,2025-06-20,quiet,4076.5,3.8,7.3,112.0
6,2026-04-17,quiet,4434.0,2.4,1.4,135.0


---
## 5. Does the separation actually hold?


In [6]:
gaps = selected.cutoff.diff().dt.days.dropna()
print(
    f"closest two cutoffs: {int(gaps.min())} days apart ({int(gaps.min()) // 7} weeks) "
    f"-- windows are {'independent' if gaps.min() >= 70 else 'OVERLAPPING'}"
)

ev_moves, qt_moves = events.max_move, quiet.max_move
print(f"\nevent windows -- mean max move {ev_moves.mean():.2f}%  (weakest: {ev_moves.min():.2f}%)")
print(f"quiet windows -- mean max move {qt_moves.mean():.2f}%  (strongest: {qt_moves.max():.2f}%)")
print(f"separation (group means): {ev_moves.mean() / qt_moves.mean():.2f}x")
print(f"cleanly ordered (every event > every quiet): {ev_moves.min() > qt_moves.max()}")

print(f"\nyears represented among events: {sorted(events.cutoff.dt.year.unique())}")

closest two cutoffs: 77 days apart (11 weeks) -- windows are independent

event windows -- mean max move 7.26%  (weakest: 7.07%)
quiet windows -- mean max move 3.36%  (strongest: 3.89%)
separation (group means): 2.16x
cleanly ordered (every event > every quiet): True

years represented among events: [np.int32(2024), np.int32(2025)]


Cleanly ordered and well separated — every event cutoff moves more than every quiet
cutoff, group means differ 2.16x. This is a materially better result than the
analogous Yahoo `CPO=F` search, which found **zero** independent 4-event
combinations at all and had to settle for a set where one quiet cutoff (5.7%)
exceeded the weakest event (3.2%). The difference is structural, not a search
artifact: MPOB's ordinary-week volatility is higher than the futures series (mean
|move| in non-roll weeks: 2.06% vs 1.13% on `CPO=F`), which spreads its moves out
instead of compressing them toward the middle — exactly what makes a clean split
possible here and not there.


---
## 6. Visual check

The chart is the audit. If an orange band looks flat, or a blue band contains a
cliff, the label is wrong and the cutoff should be swapped.


In [7]:
plot_cutoff_windows(
    svc.get_series(MPOB_WEEKLY_SERIES_ID, as_of=as_of),
    cutoffs=DEFAULT_CUTOFFS,
    horizons=13,
)

In [8]:
plot_price_history(
    svc.get_series(MPOB_WEEKLY_SERIES_ID, as_of=as_of),
    cutoffs=DEFAULT_CUTOFFS,
    start="2023-06-01",
    title="MPOB weekly (median), with the seven cutoffs",
    units="MYR per tonne",
    currency="RM",
    show_blackouts=False,
)

---
## 7. Committed selection

Frozen in `cpo.plots.DEFAULT_CUTOFFS`, so the specs, baselines, and agent
evaluation all read the same list rather than re-deriving it.


In [9]:
pd.DataFrame(
    [{"cutoff": c.date, "kind": c.kind, "why": c.label} for c in sorted(DEFAULT_CUTOFFS, key=lambda c: c.date)]
)

,cutoff,kind,why
0,2024-02-02,event,+7.7% two weeks out; +2.1% over 13 weeks
1,2024-04-19,quiet,max weekly move ahead 3.9%; -2.3% over 13 weeks
2,2024-07-26,event,+7.1% two weeks out; +16.3% over 13 weeks
3,2024-10-18,event,+7.1% two weeks out; +4.7% over 13 weeks
4,2025-01-17,event,+7.1% two weeks out; -8.7% over 13 weeks -- we...
5,2025-06-20,quiet,"max weekly move ahead 3.8% -- strongest quiet,..."
6,2026-04-17,quiet,"calmest window: max 2.4%, +1.4% over 13 weeks"


---
## 8. What this does and does not establish

**Established:** seven origins on a complete weekly grid, resolvable at every
horizon, fully independent (no two forecast windows overlap), cleanly separated
into event and quiet, each with sufficient news coverage to give an agent something
to reason over.

**Limitations to state in any writeup:**

- **All four events fall in 2024–2025; none in 2026.** The largest news-covered move
  available in 2026 is ~3%, too weak to compete for an event slot. Quiet cutoffs do
  reach into 2026 (2026-04-17).
- **Seven origins is a small sample.** Five horizons each gives 35 scored points.
  Mean CRPS differences between close predictors will not be significant. These
  cutoffs are the narrative set; a denser weekly backtest should decide which model
  is actually better.
- **Events were chosen with hindsight.** We know which weeks moved. Fine for a
  controlled comparison, not a live forecasting record.
- **18 of 734 news rows (2.5%) were malformed** and dropped — likely unescaped
  characters in the source `texts` field. Worth flagging to Jyotsna.
- **No 2022-scale shock exists in the 2024–2026 GDELT window.** The largest weekly
  move here is ~7.7%, against 30%+ during the 2022 export ban.
- **Data governance:** this notebook and its cached MPOB data are for local use
  only, per Vector's rule that MPOB requires a data-office approval inside Coder
  that has not been obtained. The notebook itself, its charts, and this frozen
  cutoff list are fine to commit and push — only the raw `data/mpob/` parquet cache
  must never leave a local machine. See `DATA.md`.

**Next:** a naive last-value baseline across these seven, then Prophet and the
Darts models, then the agent.
